<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings and methodology questions

**Finding 1 — CTR falls sharply as ranking position worsens**

The FlyRank report's Finding 3 states that click capture drops substantially as pages move from the highest search positions to deeper ranking tiers. The report shows weighted CTR of about 0.420% for the top 3 positions, compared with 0.050% for deep-ranking pages.

**Methodology question:** Because CTR is strongly related to ranking position, how was position accounted for when interpreting CTR differences? A raw comparison of CTR across pages could partly reflect ranking position rather than an independent CTR effect.

**Finding 2 — Freshness is associated with different growth/decline patterns**

Finding 4 reports that pages in different freshness windows show different growth-to-decline ratios. For example, the 31–90 day group shows a much higher growth-to-decline ratio than some older groups, although the oldest bucket contains relatively few observations.

**Methodology question:** How were pages assigned to freshness groups, and does the validation design account for differences between newer and older pages, such as baseline traffic, content age, or selection into refresh activity? Without controlling for these differences, the observed association should not automatically be interpreted as a causal effect of refreshing.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation split

The original Week-5 model used a stratified random 80/20 split. That split preserves the decline-rate distribution, but pages associated with the same client may appear in both training and validation data.

Because the public-safe starter dataset does not contain a reliable observation-date field, I cannot construct a defensible time-aware split. Instead, I perform a grouped validation by `client_id`, keeping each client entirely in either the training or validation set.

I then compare the Random Forest under the original random split and the grouped split using the same feature set and Precision@50. A reduction under grouped validation would indicate that the random split gave an optimistic estimate of performance; a similar result would indicate greater robustness across unseen client groups.

In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

import os
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

data_path = os.path.join(
    REPO_DIR,
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))


feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

X = df[feature_cols].copy()
y = (df["trend_direction"] == "down").astype(int)


def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        ))
    ])


def precision_at_k(scores, y_true, k=50):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    top_idx = np.argsort(scores)[::-1][:k]
    return y_true[top_idx].mean()


# ------------------------------------------------
# 1. Original stratified random split
# ------------------------------------------------
X_train_r, X_valid_r, y_train_r, y_valid_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()
random_model.fit(X_train_r, y_train_r)

random_scores = random_model.predict_proba(X_valid_r)[:, 1]

random_p50 = precision_at_k(
    random_scores,
    y_valid_r,
    k=50
)


# ------------------------------------------------
# 2. Grouped split by client_id
# ------------------------------------------------
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_g = X.iloc[train_idx]
X_valid_g = X.iloc[valid_idx]

y_train_g = y.iloc[train_idx]
y_valid_g = y.iloc[valid_idx]

groups_train = groups.iloc[train_idx]
groups_valid = groups.iloc[valid_idx]

group_model = make_model()
group_model.fit(X_train_g, y_train_g)

group_scores = group_model.predict_proba(X_valid_g)[:, 1]

group_p50 = precision_at_k(
    group_scores,
    y_valid_g,
    k=50
)


# ------------------------------------------------
# Comparison
# ------------------------------------------------
comparison = pd.DataFrame({
    "Validation design": [
        "Random stratified split",
        "Grouped by client_id"
    ],
    "Validation rows": [
        len(X_valid_r),
        len(X_valid_g)
    ],
    "Decline rate": [
        y_valid_r.mean(),
        y_valid_g.mean()
    ],
    "Precision@50": [
        random_p50,
        group_p50
    ]
})

print("VALIDATION COMPARISON")
print(comparison.round(3).to_string(index=False))

print("\nGrouped-split integrity checks:")
print(
    "Clients appearing in both train and validation:",
    len(set(groups_train) & set(groups_valid))
)

print(
    "Training clients:",
    groups_train.nunique()
)

print(
    "Validation clients:",
    groups_valid.nunique()
)

Rows loaded: 30000
Columns: 44
VALIDATION COMPARISON
      Validation design  Validation rows  Decline rate  Precision@50
Random stratified split             6000         0.542          0.88
   Grouped by client_id             6163         0.511          0.56

Grouped-split integrity checks:
Clients appearing in both train and validation: 0
Training clients: 25
Validation clients: 7


### Validation interpretation

The validation design materially changes the measured performance. Under the original stratified random split, the Random Forest achieved Precision@50 of 0.88. When the data were split by `client_id`, so that no client appeared in both training and validation sets, Precision@50 fell to 0.56.

This suggests that the random split gave an optimistic estimate of performance because pages from the same clients could appear on both sides of the split. The grouped result is therefore the more conservative estimate for generalization to unseen clients. I would report the model as useful for ranking support, but not claim that the 0.88 result represents expected performance on entirely new clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I repeat the leakage and privacy checks on the final feature set used in the validation experiment.

The target `trend_direction`, identifiers such as `content_id` and `client_id`, and obvious private or future-looking fields must not appear in the model inputs. `client_id` is used only to define validation groups and is not supplied to the Random Forest as a feature.

In [11]:
forbidden_exact = {
    "trend_direction",
    "content_id",
    "client_id"
}

future_like_terms = [
    "future",
    "next_",
    "after_",
    "post_",
    "outcome",
    "label"
]

private_like_terms = [
    "client",
    "url",
    "domain",
    "query"
]

exact_forbidden_found = [
    c for c in feature_cols
    if c in forbidden_exact
]

future_like_found = [
    c for c in feature_cols
    if any(term in c.lower() for term in future_like_terms)
]

private_like_found = [
    c for c in feature_cols
    if any(term in c.lower() for term in private_like_terms)
]

print("FINAL FEATURE LEAKAGE AUDIT")
print("---------------------------")
print("Features:", feature_cols)
print("Forbidden fields used:", exact_forbidden_found)
print("Future/outcome-like fields:", future_like_found)
print("Private/context-like fields:", private_like_found)

print("\nTarget excluded:",
      "trend_direction" not in feature_cols)

print("Identifiers excluded:",
      all(c not in feature_cols for c in ["content_id", "client_id"]))

print("client_id used only for grouping:",
      "client_id" in df.columns and "client_id" not in feature_cols)

FINAL FEATURE LEAKAGE AUDIT
---------------------------
Features: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count']
Forbidden fields used: []
Future/outcome-like fields: []
Private/context-like fields: []

Target excluded: True
Identifiers excluded: True
client_id used only for grouping: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Overstated version:**  
The Random Forest accurately identifies declining pages and should be used to decide which content must be refreshed.

**Safer version:**  
In the observed validation data, the Random Forest provided useful ranking signal for identifying pages associated with decline. Performance was stronger under a random split than under a client-grouped split, so the model should be treated as a directional decision-support tool rather than an automatic refresh decision system.

The model does not establish causality and does not prove that refreshing a high-scoring page will improve search performance.

In [12]:
claim_audit = {
    "uses_observed_language": True,
    "uses_directional_language": True,
    "uses_decision_support_language": True,
    "claims_causality": False
}

for check, value in claim_audit.items():
    print(f"{check}: {value}")


uses_observed_language: True
uses_directional_language: True
uses_decision_support_language: True
claims_causality: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.